# Experiment 3 - Observation-frequency distribution shift

This notebook analyzes observation-frequency distribution shift for the fixed-horizon ICU-exit task. It is intentionally self-contained and uses shared mechanics from `visualize_utils.py`; it does not train models and does not modify Experiments 1/2A/2B.

## Scientific question

Question: Does a model trained under one observation-frequency regime lose additional performance when deployed under a different observation-frequency regime, beyond the unavoidable information loss caused by having fewer observations at test time?

Observation-frequency distribution shift means the train/validation observation regime differs from the test/deployment observation regime. The key comparison is `1->r` versus `r->r`: both models are evaluated on the same structured-r sparse test observations, but only the `r->r` model was trained under that sparse regime.

This matters because absolute degradation from `1->1` to sparse testing mixes two effects: fewer test observations and train/deployment mismatch. Comparing `M(r->r)` against `M(1->r)` holds the sparse test data fixed and targets the mismatch penalty.

Caveat: `information_effect + shift_penalty` should not be read as an exact additive causal decomposition.

## Experiment definition

Primary regimes:

- `1->1`: train sampling `none`, test sampling `none`
- `1->4`: train sampling `none`, test sampling `structured`, `r=4`
- `4->4`: train sampling `structured`, `r=4`, test sampling `structured`, `r=4`
- `1->8`: train sampling `none`, test sampling `structured`, `r=8`
- `8->8`: train sampling `structured`, `r=8`, test sampling `structured`, `r=8`

The left side is the train/validation observation regime. The right side is the test/deployment observation regime. All primary runs use `timestep = 1.0`; structured thinning changes which measurements are available while keeping the downstream model representation on the same 1h grid.

The normalizer belongs to the training regime. Thus `1->4` uses the 1h-trained normalizer, while `4->4` uses the structured-r4 training normalizer. This notebook only analyzes outputs and preserves that provenance.

## Configuration

In [9]:

from pathlib import Path
import os
import re
import shlex
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from visualize_utils import (
    DEFAULT_BOOTSTRAP_SEED,
    across_seed_patient_bootstrap,
    compute_metric,
    find_selected_checkpoint,
    load_prediction_csv,
    metric_difference,
    paired_patient_bootstrap,
    paired_prediction_frame,
    prediction_metrics,
    read_validation_log,
    show_table,
    summarize_with_t_ci,
)

PROJECT_DATA_ROOT = Path(
    "/heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/"
    "physionet.org/files/mimiciv/1.0/russo"
)

EXP3_REPO_DIR = Path.cwd().parent if Path.cwd().name == "visualizers" else Path.cwd()
EXP3_PYTHON = os.environ.get("EXP3_PYTHON", "python")
EXP3_DATA_DIR = PROJECT_DATA_ROOT / "data/length-of-stay"
EXP3_NORMALIZER_DIR = PROJECT_DATA_ROOT / "normalizers"
EXP3_RESULTS_DIR = PROJECT_DATA_ROOT / "results/fixed_horizon_icu_exit"
EXP3_OUTPUT_DIR = EXP3_RESULTS_DIR / "plots"
EXP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TEST_INFERENCE = True

EXP3_NETWORK = "mimic4models/keras_models/lstm.py"
NETWORK = EXP3_NETWORK
EXP3_HORIZON = 12
EXP3_TARGET_RS = [4, 8]
EXP3_OPTIONAL_RS = [2]
EXP3_INCLUDE_OPTIONAL_RS_IF_AVAILABLE = False
EXP3_EXPECTED_MODEL_SEEDS = {0, 1, 2, 3, 4}
EXP3_EXPECTED_EPOCHS = 100
EXP3_TIMESTEP = 1.0
EXP3_REQUIRE_COMPLETE_EPOCHS = True
EXP3_ALLOW_PARTIAL_RESULTS = False
EXP3_METRICS = ["auroc", "auprc", "brier"]
EXP3_N_BOOT = int(os.environ.get("EXP3_N_BOOT", "2000"))
EXP3_BOOTSTRAP_SEED = DEFAULT_BOOTSTRAP_SEED

EXP3_EXPECTED_CONFIG = {
    "network": EXP3_NETWORK,
    "dim": 16,
    "depth": 2,
    "dropout": 0.3,
    "rec_dropout": 0.0,
    "batch_norm": False,
    "batch_size": 8,
    "timestep": EXP3_TIMESTEP,
    "target_repl_coef": 0.0,
    "l1": 0.0,
    "l2": 0.0,
    "optimizer": "adam",
    "lr": 0.001,
    "beta_1": 0.9,
    "imputation": "previous",
    "prefix": "",
}

print("Experiment 3 horizon:", EXP3_HORIZON)
print("Primary target r values:", EXP3_TARGET_RS)
print("Expected model seeds:", sorted(EXP3_EXPECTED_MODEL_SEEDS))
print("Bootstrap draws:", EXP3_N_BOOT)
print("Partial final summaries allowed:", EXP3_ALLOW_PARTIAL_RESULTS)
print("Run missing test inference:", RUN_TEST_INFERENCE)


Experiment 3 horizon: 12
Primary target r values: [4, 8]
Expected model seeds: [0, 1, 2, 3, 4]
Bootstrap draws: 2000
Partial final summaries allowed: False
Run missing test inference: True


## Run discovery and provenance

The discovery code follows the old notebook provenance chain:

`validation log -> validation-AUPRC-selected epoch -> exact checkpoint filename -> exact test prediction filename`

Matched-regime predictions must use the original filename `<checkpoint>.csv`. Shifted predictions must use an explicit `.testsample-...` suffix. The test regime is never inferred from directory name alone, and file modification time is never used.

In [10]:

def _search(pattern, text, cast=None, default=None):
    m = re.search(pattern, text)
    if m is None:
        return default
    value = m.group(1)
    return cast(value) if cast is not None else value


def parse_sampling_from_name(name):
    sample = re.search(
        r"\.sample(?P<strategy>structured|random_matched)"
        r"\.r(?P<interval>\d+)"
        r"(?:\.sseed(?P<sseed>\d+))?",
        name,
    )
    if sample is None:
        return "none", None, None
    strategy = sample.group("strategy")
    interval = int(sample.group("interval"))
    seed = sample.group("sseed")
    return strategy, interval, int(seed) if seed is not None else None


def parse_exp3_log_name(log_path):
    name = log_path.name
    strategy, interval, sampling_seed = parse_sampling_from_name(name)
    l1 = _search(r"\.L1([0-9.eE+-]+)(?=\.|$)", name, float, 0.0)
    l2 = _search(r"\.L2([0-9.eE+-]+)(?=\.|$)", name, float, 0.0)
    return {
        "filename": name,
        "horizon": _search(r"\.h(\d+)", name, int),
        "timestep": _search(r"\.ts([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float),
        "model_seed": _search(r"\.seed(\d+)(?:\.|$)", name, int),
        "train_sampling_strategy": strategy,
        "train_sampling_interval": interval,
        "train_sampling_seed": sampling_seed,
        "network": EXP3_EXPECTED_CONFIG["network"],
        "dim": _search(r"\.n(\d+)", name, int),
        "dropout": _search(r"\.d([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "rec_dropout": _search(r"\.rd([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "batch_norm": re.search(r"\.bn(?=\.|$)", name) is not None,
        "depth": _search(r"\.dep(\d+)", name, int),
        "batch_size": _search(r"\.bs(\d+)", name, int),
        "target_repl_coef": _search(r"\.trc([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "l1": l1,
        "l2": l2,
        "optimizer": EXP3_EXPECTED_CONFIG["optimizer"],
        "lr": EXP3_EXPECTED_CONFIG["lr"],
        "beta_1": EXP3_EXPECTED_CONFIG["beta_1"],
        "imputation": EXP3_EXPECTED_CONFIG["imputation"],
        "prefix": EXP3_EXPECTED_CONFIG["prefix"],
        "has_l1": l1 > 0,
        "has_l2": l2 > 0,
        "log_path": str(log_path),
        "output_dir": str(log_path.parent.parent),
    }


def effective_sampling_regime(strategy, interval, seed):
    if strategy == "none":
        return ("none",)
    if strategy == "structured":
        return ("structured", int(interval))
    if strategy == "random_matched":
        return ("random_matched", int(interval), seed)
    raise ValueError("Unknown sampling strategy {}".format(strategy))


def regimes_match(train_strategy, train_interval, train_seed, test_strategy, test_interval, test_seed):
    return effective_sampling_regime(train_strategy, train_interval, train_seed) == effective_sampling_regime(
        test_strategy, test_interval, test_seed)


def test_sampling_suffix(strategy, interval, seed):
    if strategy == "none":
        return "testsample-none"
    label = "testsample-{}-r{}".format(strategy, int(interval))
    if strategy == "random_matched":
        label += "-sseed{}".format(seed)
    return label


def expected_prediction_path(output_dir, checkpoint, train_strategy, train_interval, train_seed,
                             test_strategy, test_interval, test_seed):
    pred_dir = Path(output_dir) / "test_predictions"
    base = pred_dir / checkpoint.name
    if regimes_match(train_strategy, train_interval, train_seed, test_strategy, test_interval, test_seed):
        return Path(str(base) + ".csv")
    return Path(str(base) + ".{}.csv".format(test_sampling_suffix(test_strategy, test_interval, test_seed)))


def parse_prediction_test_regime(prediction_path, checkpoint, train_strategy, train_interval, train_seed):
    name = Path(prediction_path).name
    matched_name = checkpoint.name + ".csv"
    if name == matched_name:
        return train_strategy, train_interval, train_seed, "matched-original-filename"
    prefix = checkpoint.name + ".testsample-"
    if not (name.startswith(prefix) and name.endswith(".csv")):
        raise ValueError("Prediction file does not match checkpoint naming convention: {}".format(prediction_path))
    label = name[len(prefix):-4]
    if label == "none":
        return "none", None, None, "explicit-testsample-suffix"
    m = re.match(r"(?P<strategy>structured|random_matched)-r(?P<interval>\d+)(?:-sseed(?P<seed>\d+))?$", label)
    if m is None:
        raise ValueError("Unrecognized test-sampling suffix in {}".format(prediction_path))
    strategy = m.group("strategy")
    interval = int(m.group("interval"))
    seed = m.group("seed")
    return strategy, interval, int(seed) if seed is not None else None, "explicit-testsample-suffix"


def regime_label(train_strategy, train_interval, test_strategy, test_interval):
    left = "1" if train_strategy == "none" else str(int(train_interval))
    right = "1" if test_strategy == "none" else str(int(test_interval))
    return "{}->{}".format(left, right)


def _float_mismatch(actual, expected, tol=1e-9):
    if actual is None:
        return True
    return abs(float(actual) - float(expected)) > tol


def hyperparameter_mismatch_reason(run):
    reasons = []
    for key in ["network", "dim", "depth", "batch_size", "optimizer", "imputation", "prefix"]:
        if run.get(key) != EXP3_EXPECTED_CONFIG[key]:
            reasons.append("{}={} expected {}".format(key, run.get(key), EXP3_EXPECTED_CONFIG[key]))
    for key in ["dropout", "rec_dropout", "target_repl_coef", "l1", "l2", "timestep", "lr", "beta_1"]:
        if _float_mismatch(run.get(key), EXP3_EXPECTED_CONFIG[key]):
            reasons.append("{}={} expected {}".format(key, run.get(key), EXP3_EXPECTED_CONFIG[key]))
    if bool(run.get("batch_norm")) != bool(EXP3_EXPECTED_CONFIG["batch_norm"]):
        reasons.append("batch_norm={} expected {}".format(run.get("batch_norm"), EXP3_EXPECTED_CONFIG["batch_norm"]))
    if run.get("model_seed") not in EXP3_EXPECTED_MODEL_SEEDS:
        reasons.append("model_seed={} not in expected {}".format(run.get("model_seed"), sorted(EXP3_EXPECTED_MODEL_SEEDS)))
    return "; ".join(reasons)


def normalizer_candidates_for_train_regime(train_strategy, train_interval, train_seed):
    if train_strategy == "none":
        pattern = "fixed_horizon_icu_exit_ts:{:.2f}_impute:{}_start:zero_masks:True_n:*.normalizer".format(
            EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    elif train_strategy == "structured":
        pattern = (
            "fixed_horizon_icu_exit_sampling:structured_r:{}_ts:{:.2f}"
            "_impute:{}_start:zero_masks:True_n:*.normalizer"
        ).format(int(train_interval), EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    elif train_strategy == "random_matched":
        pattern = (
            "fixed_horizon_icu_exit_sampling:random_matched_r:{}_seed:{}_ts:{:.2f}"
            "_impute:{}_start:zero_masks:True_n:*.normalizer"
        ).format(int(train_interval), int(train_seed), EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    else:
        raise ValueError("Unknown train strategy {}".format(train_strategy))
    return sorted(EXP3_NORMALIZER_DIR.glob(pattern))


def resolve_normalizer_for_train_regime(train_strategy, train_interval, train_seed):
    candidates = normalizer_candidates_for_train_regime(train_strategy, train_interval, train_seed)
    if len(candidates) != 1:
        raise RuntimeError(
            "Expected exactly one training-regime normalizer for {} but found {}: {}".format(
                effective_sampling_regime(train_strategy, train_interval, train_seed),
                len(candidates), [str(x) for x in candidates]))
    return candidates[0]


def quote_command(cmd):
    return " ".join(shlex.quote(str(x)) for x in cmd)


def build_inference_command_parts(row):
    normalizer_path = row.get("normalizer_path")
    if not normalizer_path:
        normalizer_path = str(resolve_normalizer_for_train_regime(
            row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"]))
    cmd = [
        EXP3_PYTHON, "-m", "mimic4models.fixed_horizon_icu_exit.main",
        "--mode", "test",
        "--network", row["network"],
        "--data", str(EXP3_DATA_DIR),
        "--normalizer_dir", str(EXP3_NORMALIZER_DIR),
        "--normalizer_state", str(normalizer_path),
        "--output_dir", row["output_dir"],
        "--load_state", row["checkpoint_path"],
        "--horizon", str(int(row["horizon"])),
        "--timestep", str(row["timestep"]),
        "--seed", str(int(row["model_seed"])),
        "--dim", str(int(row["dim"])),
        "--depth", str(int(row["depth"])),
        "--dropout", str(row["dropout"]),
        "--rec_dropout", str(row["rec_dropout"]),
        "--batch_size", str(int(row["batch_size"])),
        "--target_repl_coef", str(row["target_repl_coef"]),
        "--l1", str(row["l1"]),
        "--l2", str(row["l2"]),
        "--optimizer", row["optimizer"],
        "--lr", str(row["lr"]),
        "--beta_1", str(row["beta_1"]),
        "--imputation", row["imputation"],
        "--prefix", row["prefix"],
        "--sampling_strategy", row["train_sampling_strategy"],
    ]
    if bool(row.get("batch_norm")):
        cmd += ["--batch_norm", "True"]
    if row["train_sampling_strategy"] != "none":
        cmd += ["--sampling_interval", str(int(row["train_sampling_interval"]))]
    if row["train_sampling_seed"] is not None and not pd.isnull(row["train_sampling_seed"]):
        cmd += ["--sampling_seed", str(int(row["train_sampling_seed"]))]
    if not regimes_match(row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"],
                         row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"]):
        cmd += ["--test_sampling_strategy", row["test_sampling_strategy"]]
        if row["test_sampling_strategy"] != "none":
            cmd += ["--test_sampling_interval", str(int(row["test_sampling_interval"]))]
        if row["test_sampling_seed"] is not None and not pd.isnull(row["test_sampling_seed"]):
            cmd += ["--test_sampling_seed", str(int(row["test_sampling_seed"]))]
    return cmd


def build_missing_inference_command(row):
    return quote_command(build_inference_command_parts(row))


def run_missing_test_inference(row):
    cmd = build_inference_command_parts(row)
    print("\nRunning Experiment 3 missing test inference")
    print("  regime:", row["regime"])
    print("  training regime:", effective_sampling_regime(row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"]))
    print("  test regime:", effective_sampling_regime(row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"]))
    print("  model seed:", row["model_seed"])
    print("  selected validation epoch:", row["selected_val_epoch"])
    print("  checkpoint:", row["checkpoint_path"])
    print("  normalizer:", row["normalizer_path"])
    print("  expected prediction:", row["expected_prediction_path"])
    print("  command:", quote_command(cmd))
    completed = subprocess.run(cmd, cwd=str(EXP3_REPO_DIR))
    if completed.returncode != 0:
        raise RuntimeError("Test inference failed with return code {} for {}".format(
            completed.returncode, row["expected_prediction_path"]))
    if not Path(row["expected_prediction_path"]).exists():
        raise RuntimeError("Test inference completed but exact prediction file is missing: {}".format(
            row["expected_prediction_path"]))


In [11]:

def structured_logs_available_for_r(r):
    log_dir = EXP3_RESULTS_DIR / "structured" / "{}h".format(EXP3_HORIZON) / "{}h".format(r) / "keras_logs"
    return log_dir.exists() and any(log_dir.glob("*.csv"))


exp3_rs = list(EXP3_TARGET_RS)
if EXP3_INCLUDE_OPTIONAL_RS_IF_AVAILABLE:
    for optional_r in EXP3_OPTIONAL_RS:
        if optional_r not in exp3_rs and structured_logs_available_for_r(optional_r):
            exp3_rs.append(optional_r)
exp3_rs = sorted(exp3_rs)
print("Experiment 3 r values considered:", exp3_rs)

candidate_log_paths = sorted((EXP3_RESULTS_DIR / "{}h".format(EXP3_HORIZON) / "keras_logs").glob("*.csv"))
for r in exp3_rs:
    candidate_log_paths.extend(sorted(
        (EXP3_RESULTS_DIR / "structured" / "{}h".format(EXP3_HORIZON) / "{}h".format(r) / "keras_logs").glob("*.csv")
    ))

train_status_rows = []
train_excluded_rows = []
train_runs = {}

for log_path in sorted(set(candidate_log_paths)):
    parsed = parse_exp3_log_name(log_path)
    if parsed["horizon"] != EXP3_HORIZON:
        continue
    train_strategy = parsed["train_sampling_strategy"]
    train_interval = parsed["train_sampling_interval"]
    is_dense_train = train_strategy == "none" and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6
    is_structured_train = (train_strategy == "structured" and train_interval in exp3_rs
                           and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6)
    if not (is_dense_train or is_structured_train):
        row = dict(parsed)
        row["exclude_reason"] = "outside Experiment 3 train regimes"
        train_excluded_rows.append(row)
        continue

    mismatch = hyperparameter_mismatch_reason(parsed)
    if mismatch:
        row = dict(parsed)
        row["exclude_reason"] = mismatch
        train_excluded_rows.append(row)
        continue

    val = read_validation_log(log_path, expected_epochs=EXP3_EXPECTED_EPOCHS)
    if val is None:
        row = dict(parsed)
        row["exclude_reason"] = "empty validation log"
        train_excluded_rows.append(row)
        continue
    parsed.update(val)
    parsed["selected_val_epoch"] = parsed["selected_validation_epoch"]
    parsed["selected_val_auprc"] = parsed["validation_auprc"]
    parsed["selected_val_auroc"] = parsed["validation_auroc"]
    checkpoint = find_selected_checkpoint(log_path, parsed["selected_validation_epoch"])
    parsed["checkpoint_path"] = None if checkpoint is None else str(checkpoint)
    parsed["checkpoint_name"] = None if checkpoint is None else checkpoint.name
    normalizer = resolve_normalizer_for_train_regime(
        train_strategy, train_interval, parsed["train_sampling_seed"])
    parsed["normalizer_path"] = str(normalizer)
    parsed["normalizer_candidates"] = [str(normalizer)]
    train_status_rows.append(dict(parsed))

    if EXP3_REQUIRE_COMPLETE_EPOCHS and not parsed["complete"]:
        row = dict(parsed)
        row["exclude_reason"] = "incomplete epoch set; require exact 0..99"
        train_excluded_rows.append(row)
        continue
    if checkpoint is None:
        row = dict(parsed)
        row["exclude_reason"] = "missing validation-selected checkpoint"
        train_excluded_rows.append(row)
        continue

    key = (train_strategy, train_interval, parsed["model_seed"])
    if key in train_runs:
        raise RuntimeError("Duplicate train-run identity {}:\n{}\n{}".format(
            key, train_runs[key]["log_path"], parsed["log_path"]))
    train_runs[key] = parsed

exp3_train_status = pd.DataFrame(train_status_rows)
exp3_train_excluded = pd.DataFrame(train_excluded_rows)
show_table("Experiment 3 train-run status", exp3_train_status, max_rows=200)
show_table("Experiment 3 excluded train logs", exp3_train_excluded, max_rows=200)


Experiment 3 r values considered: [4, 8]

Experiment 3 train-run status


,batch_norm,batch_size,best_csv_epoch,best_val_auprc,best_val_epoch,beta_1,checkpoint_epoch,checkpoint_name,checkpoint_path,complete,...,target_repl_coef,timestep,train_sampling_interval,train_sampling_seed,train_sampling_strategy,val_auprc,val_auroc,val_auroc_at_best_auprc,validation_auprc,validation_auroc
0,False,8,93,0.469834,93,0.9,94,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed0.epoch...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,NaN,None,none,0.469834,0.730083,0.730083,0.469834,0.730083
1,False,8,44,0.476230,44,0.9,45,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed1.epoch...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,NaN,None,none,0.476230,0.731720,0.731720,0.476230,0.731720
2,False,8,91,0.471504,91,0.9,92,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed2.epoch...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,NaN,None,none,0.471504,0.736019,0.736019,0.471504,0.736019
3,False,8,92,0.471724,92,0.9,93,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed3.epoch...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,NaN,None,none,0.471724,0.732311,0.732311,0.471724,0.732311
4,False,8,86,0.479264,86,0.9,87,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed4.epoch...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,NaN,None,none,0.479264,0.734050,0.734050,0.479264,0.734050
5,False,8,85,0.449649,85,0.9,86,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.samplestruc...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,4.0,None,structured,0.449649,0.720474,0.720474,0.449649,0.720474
6,False,8,88,0.454407,88,0.9,89,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.samplestruc...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,4.0,None,structured,0.454407,0.721999,0.721999,0.454407,0.721999
7,False,8,88,0.454703,88,0.9,89,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.samplestruc...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,4.0,None,structured,0.454703,0.723755,0.723755,0.454703,0.723755
8,False,8,17,0.433833,17,0.9,18,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.samplestruc...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,False,...,0.0,1.0,4.0,None,structured,0.433833,0.704735,0.704735,0.433833,0.704735
9,False,8,99,0.445862,99,0.9,100,k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.samplestruc...,/heinz-georgenas/users/mingzhul/Simultaneous-E...,True,...,0.0,1.0,8.0,None,structured,0.445862,0.708989,0.708989,0.445862,0.708989



Experiment 3 excluded train logs


,batch_norm,batch_size,best_csv_epoch,best_val_auprc,best_val_epoch,beta_1,checkpoint_epoch,checkpoint_name,checkpoint_path,complete,...,target_repl_coef,timestep,train_sampling_interval,train_sampling_seed,train_sampling_strategy,val_auprc,val_auroc,val_auroc_at_best_auprc,validation_auprc,validation_auroc
0,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,12.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
1,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,12.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
2,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,12.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
3,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,12.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
4,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,12.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
5,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,2.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
6,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,2.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
7,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,2.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
8,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,2.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN
9,False,8,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,...,0.0,2.0,NaN,None,none,NaN,NaN,NaN,NaN,NaN


In [ ]:

def _load_prediction_into_row(row, prediction_path, checkpoint):
    parsed_test_strategy, parsed_test_interval, parsed_test_seed, name_mode = parse_prediction_test_regime(
        prediction_path, checkpoint,
        row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"])
    expected_effective = effective_sampling_regime(
        row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"])
    parsed_effective = effective_sampling_regime(parsed_test_strategy, parsed_test_interval, parsed_test_seed)
    if parsed_effective != expected_effective:
        raise RuntimeError("Prediction test-regime suffix mismatch for {}: parsed {} expected {}".format(
            prediction_path, parsed_effective, expected_effective))
    pred_df = load_prediction_csv(prediction_path)
    metrics = prediction_metrics(pred_df)
    row.update(metrics)
    row["prediction_path"] = str(prediction_path)
    row["prediction_status"] = "found"
    row["prediction_exists"] = True
    row["prediction_name_mode"] = name_mode
    row["missing_reason"] = None


def add_planned_eval(planned_rows, train_run, test_strategy, test_interval, test_seed):
    label = regime_label(train_run["train_sampling_strategy"], train_run["train_sampling_interval"],
                         test_strategy, test_interval)
    row = dict(train_run)
    row.update({
        "test_sampling_strategy": test_strategy,
        "test_sampling_interval": test_interval,
        "test_sampling_seed": test_seed,
        "regime": label,
        "target_r": test_interval if test_strategy != "none" else 1,
    })
    checkpoint = Path(row["checkpoint_path"])
    prediction_path = expected_prediction_path(
        row["output_dir"], checkpoint,
        row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"],
        test_strategy, test_interval, test_seed,
    )
    row["expected_prediction_path"] = str(prediction_path)
    row["prediction_path"] = None
    row["prediction_status"] = "missing"
    row["prediction_exists"] = prediction_path.exists()
    row["prediction_name_mode"] = None
    row["missing_reason"] = None
    row["suggested_inference_command"] = build_missing_inference_command(row)
    if prediction_path.exists():
        _load_prediction_into_row(row, prediction_path, checkpoint)
    else:
        row["missing_reason"] = "missing exact prediction for selected checkpoint/test regime"
        if RUN_TEST_INFERENCE:
            run_missing_test_inference(row)
            row["prediction_exists"] = prediction_path.exists()
            _load_prediction_into_row(row, prediction_path, checkpoint)
    planned_rows.append(row)


def assert_exp3_training_provenance(run_status):
    if run_status is None or len(run_status) == 0:
        return
    for seed in sorted(run_status["model_seed"].dropna().astype(int).unique().tolist()):
        dense = run_status[(run_status["model_seed"].astype(int) == seed) &
                           (run_status["regime"].isin(["1->1", "1->4", "1->8"]))]
        if len(dense):
            checkpoints = set(dense["checkpoint_path"].dropna().tolist())
            if len(checkpoints) != 1:
                raise RuntimeError("Dense regimes do not share one selected checkpoint for seed {}: {}".format(
                    seed, sorted(checkpoints)))
            if set(dense["train_sampling_strategy"].tolist()) != {"none"}:
                raise RuntimeError("Dense shifted regimes changed training sampling for seed {}".format(seed))
        for r in exp3_rs:
            regime = "{}->{}".format(r, r)
            rows = run_status[(run_status["model_seed"].astype(int) == seed) & (run_status["regime"] == regime)]
            if len(rows) == 0:
                continue
            for _, row in rows.iterrows():
                if row["train_sampling_strategy"] != "structured" or int(row["train_sampling_interval"]) != int(r):
                    raise RuntimeError("{} must use structured-r{} training checkpoint for seed {}".format(
                        regime, r, seed))


planned_rows = []
for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
    dense_key = ("none", None, seed)
    dense_run = train_runs.get(dense_key)
    if dense_run is not None:
        add_planned_eval(planned_rows, dense_run, "none", None, None)
        for r in exp3_rs:
            add_planned_eval(planned_rows, dense_run, "structured", r, None)
    for r in exp3_rs:
        structured_key = ("structured", r, seed)
        structured_run = train_runs.get(structured_key)
        if structured_run is not None:
            add_planned_eval(planned_rows, structured_run, "structured", r, structured_run["train_sampling_seed"])

exp3_run_status = pd.DataFrame(planned_rows)
if len(exp3_run_status):
    exp3_run_status = exp3_run_status.sort_values(["regime", "model_seed"]).reset_index(drop=True)
    assert_exp3_training_provenance(exp3_run_status)

if len(exp3_run_status):
    dup_cols = ["horizon", "regime", "model_seed", "test_sampling_strategy", "test_sampling_interval"]
    dup = exp3_run_status[exp3_run_status.duplicated(dup_cols, keep=False)]
    if len(dup):
        raise RuntimeError("Unexpected duplicate Experiment 3 identities:\n{}".format(
            dup[dup_cols + ["log_path", "expected_prediction_path"]]))

missing_mask = exp3_run_status["prediction_status"] != "found" if len(exp3_run_status) else []
exp3_missing_predictions = exp3_run_status[missing_mask].copy() if len(exp3_run_status) else pd.DataFrame()

exp3_run_results = exp3_run_status[exp3_run_status["prediction_status"] == "found"].copy() if len(exp3_run_status) else pd.DataFrame()
exp3_predictions = {}
if len(exp3_run_results):
    for _, row in exp3_run_results.iterrows():
        key = (int(row["horizon"]), row["regime"], int(row["model_seed"]))
        if key in exp3_predictions:
            raise RuntimeError("Duplicate prediction identity {}".format(key))
        exp3_predictions[key] = load_prediction_csv(row["prediction_path"])

provenance_cols = [
    "regime", "horizon", "model_seed",
    "train_sampling_strategy", "train_sampling_interval", "train_sampling_seed",
    "test_sampling_strategy", "test_sampling_interval", "test_sampling_seed",
    "network", "dim", "depth", "dropout", "rec_dropout", "batch_size", "timestep",
    "target_repl_coef", "l1", "l2", "optimizer", "lr", "beta_1", "imputation", "prefix",
    "n_epochs", "complete", "selected_val_epoch", "selected_val_auprc", "selected_val_auroc",
    "normalizer_path", "checkpoint_path", "expected_prediction_path", "prediction_path",
    "prediction_exists", "prediction_status", "prediction_name_mode",
    "number_of_stays", "number_of_patients",
]
show_table("Experiment 3 compact run-status table", exp3_run_status[[c for c in provenance_cols if c in exp3_run_status.columns]], max_rows=300)
show_table("Experiment 3 missing predictions / incomplete provenance", exp3_missing_predictions[[c for c in provenance_cols + ["missing_reason", "suggested_inference_command"] if c in exp3_missing_predictions.columns]], max_rows=300)
show_table("Experiment 3 included exact prediction rows", exp3_run_results[[c for c in provenance_cols + ["test_auroc", "test_auprc", "test_brier"] if c in exp3_run_results.columns]], max_rows=300)

if len(exp3_run_results):
    for regime, group in exp3_run_results.groupby("regime"):
        observed = set(group["model_seed"].astype(int).tolist())
        if observed != EXP3_EXPECTED_MODEL_SEEDS:
            print("PARTIAL SEED SET for {}: observed={} missing={} extra={}".format(
                regime, sorted(observed), sorted(EXP3_EXPECTED_MODEL_SEEDS - observed),
                sorted(observed - EXP3_EXPECTED_MODEL_SEEDS)))



Running Experiment 3 missing test inference
  regime: 1->4
  training regime: ('none',)
  test regime: ('structured', 4)
  model seed: 0
  selected validation epoch: 93
  checkpoint: /heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/physionet.org/files/mimiciv/1.0/russo/results/fixed_horizon_icu_exit/12h/keras_states/k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed0.epoch94.test0.5119056583997604.state
  normalizer: /heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/physionet.org/files/mimiciv/1.0/russo/normalizers/fixed_horizon_icu_exit_ts:1.00_impute:previous_start:zero_masks:True_n:21860.normalizer
  expected prediction: /heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/physionet.org/files/mimiciv/1.0/russo/results/fixed_horizon_icu_exit/12h/test_predictions/k_lstm.n16.d0.3.dep2.h12.bs8.ts1.0.seed0.epoch94.test0.5119056583997604.state.testsample-structured-r4.csv
  command: python -m mimic4models.fixed_horizon_icu_exit.main --mode test --network mimic4models/keras_models/lstm.py 

## Absolute test performance

Question: How does performance differ across train/test observation regimes?

Quantity: The x-axis is train/test regime (`1->1`, `1->4`, `4->4`, `1->8`, `8->8`). The y-axis is test AUROC or test AUPRC from the validation-AUPRC-selected checkpoint. Error bars are mean across model seeds +/- 95% Student-t CI; if `n < 2`, the CI is NaN, not zero-width.

Evidence we are looking for: `1->r` worse than `r->r` suggests a mismatch penalty.

Caveat: Absolute differences alone do not isolate the shift effect unless the test regime is the same.

In [ ]:
def complete_seed_groups(df, group_cols, seed_col="model_seed"):
    if df is None or len(df) == 0:
        return df
    if EXP3_ALLOW_PARTIAL_RESULTS:
        print("Using partial exploratory results because EXP3_ALLOW_PARTIAL_RESULTS=True.")
        return df.copy()
    keep = []
    for keys, group in df.groupby(list(group_cols)):
        observed = set(group[seed_col].astype(int).tolist())
        if observed == EXP3_EXPECTED_MODEL_SEEDS:
            keep.append(group)
        else:
            print("Excluding incomplete final group {} observed={} missing={} extra={}".format(
                keys, sorted(observed), sorted(EXP3_EXPECTED_MODEL_SEEDS - observed),
                sorted(observed - EXP3_EXPECTED_MODEL_SEEDS)))
    if not keep:
        return df.iloc[0:0].copy()
    return pd.concat(keep, ignore_index=True)


exp3_absolute_runs = complete_seed_groups(exp3_run_results, ["regime"])
exp3_absolute_summary_parts = []
for metric_col in ["test_auroc", "test_auprc", "test_brier"]:
    if len(exp3_absolute_runs):
        metric_summary = summarize_with_t_ci(exp3_absolute_runs, ["regime"], metric_col)
        metric_summary.insert(1, "metric", metric_col)
        exp3_absolute_summary_parts.append(metric_summary)
exp3_absolute_summary = pd.concat(exp3_absolute_summary_parts, ignore_index=True) if exp3_absolute_summary_parts else pd.DataFrame()
show_table("Experiment 3 absolute test performance mean +/- 95% Student-t CI", exp3_absolute_summary, max_rows=100)


def plot_absolute_metric(metric_col, ylabel, output_name):
    if len(exp3_absolute_runs) == 0:
        print("Skipping {}: no complete seed-set absolute results.".format(output_name))
        return
    summary = summarize_with_t_ci(exp3_absolute_runs, ["regime"], metric_col)
    order = ["1->1"]
    for r in exp3_rs:
        order += ["1->{}".format(r), "{}->{}".format(r, r)]
    summary["order"] = summary["regime"].apply(lambda x: order.index(x) if x in order else 999)
    summary = summary.sort_values("order")
    x = np.arange(len(summary))
    y = summary["mean"].values
    yerr = summary["ci95"].values
    plt.figure(figsize=(8, 4.8))
    plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, color="black")
    for i, regime in enumerate(summary["regime"].tolist()):
        raw = exp3_absolute_runs[exp3_absolute_runs["regime"] == regime]
        plt.scatter(np.repeat(i, len(raw)), raw[metric_col].values, alpha=0.45, s=24)
    plt.xticks(x, summary["regime"].tolist())
    plt.ylabel(ylabel)
    plt.xlabel("train -> test observation regime")
    plt.title("Experiment 3 {} by train/test regime\nmean across model seeds +/- 95% Student-t CI".format(ylabel))
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    output_path = EXP3_OUTPUT_DIR / output_name
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)


plot_absolute_metric("test_auroc", "Test AUROC", "experiment3_test_auroc_by_regime_ci95.png")
plot_absolute_metric("test_auprc", "Test AUPRC", "experiment3_test_auprc_by_regime_ci95.png")

Excluding incomplete final group 1->1 observed=[0, 1] missing=[2, 3, 4] extra=[]

Experiment 3 absolute test performance mean +/- 95% Student-t CI
<none>
Skipping experiment3_test_auroc_by_regime_ci95.png: no complete seed-set absolute results.
Skipping experiment3_test_auprc_by_regime_ci95.png: no complete seed-set absolute results.


## Information effect

Question: How much performance is associated with operating under a lower-frequency observation regime when the model is trained for that regime?

Quantity: `information_effect(r) = M(1->1) - M(r->r)` for AUROC, AUPRC, and Brier. Positive means the left/reference condition is better; for Brier, the sign is reversed because lower Brier is better.

Evidence we are looking for: Positive values indicate lower-frequency operation is associated with lower performance even when the model is trained for that lower-frequency regime.

Caveat: This is descriptive/supporting analysis and does not isolate train/test mismatch.

In [ ]:
information_rows = []
if len(exp3_run_results):
    by_key = {
        (row["regime"], int(row["model_seed"])): row
        for _, row in exp3_run_results.iterrows()
    }
    for r in exp3_rs:
        for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
            dense = by_key.get(("1->1", seed))
            sparse = by_key.get(("{}->{}".format(r, r), seed))
            if dense is None or sparse is None:
                continue
            for metric in EXP3_METRICS:
                col = "test_{}".format(metric)
                left_value = dense[col]
                right_value = sparse[col]
                information_rows.append({
                    "r": r,
                    "model_seed": seed,
                    "metric": metric,
                    "left_regime": "1->1",
                    "right_regime": "{}->{}".format(r, r),
                    "left_metric": left_value,
                    "right_metric": right_value,
                    "difference": metric_difference(left_value, right_value, metric),
                })
exp3_information_effects = pd.DataFrame(information_rows)
show_table("Experiment 3 seed-level information effects", exp3_information_effects, max_rows=200)

exp3_information_summary_parts = []
if len(exp3_information_effects):
    complete_info = complete_seed_groups(exp3_information_effects, ["r", "metric"])
    exp3_information_summary = summarize_with_t_ci(complete_info, ["r", "metric"], "difference")
else:
    exp3_information_summary = pd.DataFrame()
show_table("Experiment 3 information effect summary", exp3_information_summary, max_rows=100)


Experiment 3 seed-level information effects
<none>

Experiment 3 information effect summary
<none>


## Frequency-shift penalty

Question: Does training under the wrong observation regime create additional deployment loss?

Quantity: `shift_penalty(r) = M(r->r) - M(1->r)`. The x-axis is target test frequency `r`; the y-axis is the shift penalty. Positive means the sparse-trained model performs better than the dense-trained model on the same sparse structured-r test observations. For Brier, the sign is reversed so positive still means the left/reference condition is better.

Evidence we are looking for: A positive difference, reproducible across model seeds, especially for `r=4` or `r=8`, with positive paired-bootstrap effect.

Caveat: This demonstrates sensitivity to observation-regime mismatch but does not by itself identify the exact mechanism.

In [ ]:

def verify_same_test_examples(left_key, right_key):
    paired = paired_prediction_frame(exp3_predictions[left_key], exp3_predictions[right_key])
    return int(len(paired)), int(paired["patient_id"].nunique())


shift_rows = []
if len(exp3_run_results):
    available_keys = set(exp3_predictions.keys())
    for r in exp3_rs:
        left_regime = "{}->{}".format(r, r)
        right_regime = "1->{}".format(r)
        for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
            left_key = (EXP3_HORIZON, left_regime, seed)
            right_key = (EXP3_HORIZON, right_regime, seed)
            if left_key not in available_keys or right_key not in available_keys:
                continue
            paired_n_stays, paired_n_patients = verify_same_test_examples(left_key, right_key)
            for metric in EXP3_METRICS:
                result = paired_patient_bootstrap(
                    exp3_predictions[left_key],
                    exp3_predictions[right_key],
                    metric=metric,
                    n_boot=EXP3_N_BOOT,
                    random_state=(EXP3_BOOTSTRAP_SEED + int(r) * 1000 + int(seed) * 10 + EXP3_METRICS.index(metric)),
                )
                shift_rows.append({
                    "r": r,
                    "model_seed": seed,
                    "metric": metric,
                    "left_regime": left_regime,
                    "right_regime": right_regime,
                    "difference": result["difference"],
                    "left_metric": result["left_metric"],
                    "right_metric": result["right_metric"],
                    "patient_bootstrap_ci_low": result["ci_low"],
                    "patient_bootstrap_ci_high": result["ci_high"],
                    "n_patients": result["n_patients"],
                    "n_stays": result["n_stays"],
                    "verified_same_test_examples": True,
                    "verified_same_n_stays": paired_n_stays,
                    "verified_same_n_patients": paired_n_patients,
                    "n_boot_valid": result["n_boot_valid"],
                })
exp3_shift_contrasts = pd.DataFrame(shift_rows)
show_table("Experiment 3 seed-level shift penalties with paired patient-bootstrap CI", exp3_shift_contrasts, max_rows=200)

summary_rows = []
if len(exp3_shift_contrasts):
    for keys, group in exp3_shift_contrasts.groupby(["r", "metric", "left_regime", "right_regime"]):
        r, metric, left_regime, right_regime = keys
        seeds = sorted(group["model_seed"].astype(int).unique().tolist())
        if set(seeds) != EXP3_EXPECTED_MODEL_SEEDS and not EXP3_ALLOW_PARTIAL_RESULTS:
            print("Skipping final shift summary for r={} metric={} because seed set is partial: observed={} missing={}".format(
                r, metric, seeds, sorted(EXP3_EXPECTED_MODEL_SEEDS - set(seeds))))
            continue
        bootstrap_keys = [
            (seed, (EXP3_HORIZON, left_regime, seed), (EXP3_HORIZON, right_regime, seed))
            for seed in seeds
        ]
        boot = across_seed_patient_bootstrap(
            exp3_predictions,
            bootstrap_keys,
            metric=metric,
            n_boot=EXP3_N_BOOT,
            random_state=(EXP3_BOOTSTRAP_SEED + int(r) * 1000 + EXP3_METRICS.index(metric)),
        )
        diffs = group["difference"].astype(float).values
        summary_rows.append({
            "r": r,
            "metric": metric,
            "n_model_seeds": int(len(seeds)),
            "mean_seed_level_difference": float(np.mean(diffs)),
            "sd_across_model_seeds": float(np.std(diffs, ddof=1)) if len(diffs) > 1 else np.nan,
            "patient_bootstrap_ci_low": boot["patient_bootstrap_ci_low"],
            "patient_bootstrap_ci_high": boot["patient_bootstrap_ci_high"],
            "left_regime": left_regime,
            "right_regime": right_regime,
            "complete_seed_set": set(seeds) == EXP3_EXPECTED_MODEL_SEEDS,
        })
exp3_across_seed_summary = pd.DataFrame(summary_rows)
if len(exp3_across_seed_summary):
    exp3_across_seed_summary = exp3_across_seed_summary.sort_values(["metric", "r"]).reset_index(drop=True)
show_table("Experiment 3 across-seed shift-penalty summary", exp3_across_seed_summary, max_rows=100)



Experiment 3 seed-level shift penalties with paired patient-bootstrap CI
<none>

Experiment 3 across-seed shift-penalty summary
<none>


In [ ]:
def plot_shift_penalty(metric, ylabel, output_name):
    if len(exp3_across_seed_summary) == 0:
        print("Skipping {}: no complete across-seed shift summaries.".format(output_name))
        return
    sub = exp3_across_seed_summary[exp3_across_seed_summary["metric"] == metric].sort_values("r")
    if len(sub) == 0:
        print("Skipping {}: no rows for metric {}.".format(output_name, metric))
        return
    x = np.arange(len(sub))
    y = sub["mean_seed_level_difference"].values
    lo = sub["patient_bootstrap_ci_low"].values
    hi = sub["patient_bootstrap_ci_high"].values
    yerr = np.vstack([y - lo, hi - y])
    plt.figure(figsize=(6.5, 4.5))
    plt.axhline(0, color="gray", linewidth=1, linestyle="--")
    plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=5, color="black")
    for i, r in enumerate(sub["r"].tolist()):
        raw = exp3_shift_contrasts[(exp3_shift_contrasts["metric"] == metric) & (exp3_shift_contrasts["r"] == r)]
        plt.scatter(np.repeat(i, len(raw)), raw["difference"].values, alpha=0.45, s=24)
    plt.xticks(x, ["r={}".format(int(r)) for r in sub["r"].tolist()])
    plt.xlabel("target test frequency")
    plt.ylabel(ylabel)
    plt.title("Experiment 3 {} shift penalty\nM(r->r) - M(1->r), 95% paired patient-bootstrap CI".format(metric.upper()))
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    output_path = EXP3_OUTPUT_DIR / output_name
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)


plot_shift_penalty("auroc", "AUROC shift penalty (left better +)", "experiment3_shift_penalty_auroc_bootstrap_ci95.png")
plot_shift_penalty("auprc", "AUPRC shift penalty (left better +)", "experiment3_shift_penalty_auprc_bootstrap_ci95.png")

Skipping experiment3_shift_penalty_auroc_bootstrap_ci95.png: no complete across-seed shift summaries.
Skipping experiment3_shift_penalty_auprc_bootstrap_ci95.png: no complete across-seed shift summaries.


## Across-seed summary

The table above reports `r`, metric, number of model seeds, mean seed-level difference, SD across model seeds, and paired patient-bootstrap 95% CI. The bootstrap samples patients with replacement; all stays for sampled patients are included. When multiple model seeds are available, each bootstrap replicate averages the paired effect across seeds.

## Go/no-go interpretation

Evidence supporting the methods direction: `M(r->r) > M(1->r)` reproducibly across model seeds, especially for `r=4` or `r=8`, with a positive paired-bootstrap effect. This means the dense-trained model suffers an avoidable penalty when deployed under a sparse observation regime.

Evidence against this specific methods direction: `M(r->r) ~= M(1->r)`. In that case, most of the degradation is likely explained by loss of available information, not train/deployment observation-frequency mismatch.

Do not force a methods-paper conclusion if the effect is weak.

## Source-notebook mapping

`visualize.ipynb` remains unchanged for manual review. This new Experiment 3 notebook does not move or delete old Experiment 1/2A/2B analyses. It reuses the shared mechanics that were duplicated in the source notebook:

- source cells 1-2: uncertainty conventions, metric definitions, validation-log checkpoint selection, exact prediction loading, Student-t summaries;
- source cell 18: patient ID extraction and paired patient bootstrap logic;
- source cells 20-25: fixed-horizon run parsing, strict provenance checks, exact checkpoint-to-prediction matching, run-level summaries, and across-seed paired bootstrap structure.

No old analysis section is intentionally omitted from Experiment 3; old Experiment 1/2A/2B analyses are out of scope for this add-only notebook and remain in `visualize.ipynb`.